# Workflow: Evaluator-Optimizer

## Load Keys from Env.

In [9]:
from dotenv import load_dotenv
load_dotenv()

True

In [10]:
import os
from cohere import ClientV2

co = ClientV2(api_key=os.environ["COHERE_API_KEY"])

MODEL_NAME = "command-a-03-2025"

## Define Helper for LLM calls

In [11]:
def llm(prompt: str, system: str = "") -> str:
    msgs = []
    if system:
        msgs.append(
            {
                "role": "system",
                "content": system,
            }
        )
    msgs.append(
        {
            "role": "user",
            "content": prompt,
        }
    )

    return (
        co.chat(
            model=MODEL_NAME,
            messages=msgs,
        )
        .message.content[0]
        .text.strip()
    )

## Optimizer

In [12]:
def generate(brief: str, feedback: str = "") -> str:
    prompt = f"Write a two sentence product description for {brief}"
    if feedback:
        prompt += f"\nRevise to address this feedback:\n{feedback}"
    return llm(
        prompt,
        system="You are a copywriter. Write punchy, concrete copy. Avoid cliches like 'revolutionize' or 'game-changer'.",
    )

## Evaluator

In [13]:
import json

def evaluate(text: str) -> dict:
    raw = llm(
        f"Evaluate this product copy:\n\n{text!r}\n\n"
        "Criteria:\n"
        "1. No clichés ('revolutionize', 'game-changer', 'next-gen', 'cutting-edge').\n"
        "2. Contains at least one concrete number or fact.\n"
        "3. Two sentences exactly.\n\n"
        'Respond ONLY in JSON: {"approved": bool, "feedback": "..."}'
    )
    
    # Handle code fences.
    raw = raw.strip().strip("`")
    if raw.startswith("json"):
        raw = raw[:4].strip()
    return json.loads(raw)

## Max Feedback Loops

In [14]:
MAX_ROUNDS = 3

## Execute With Input

In [15]:
brief = (
    "a $39 smart water bottle that tracks hydration and glows when you need to drink"
)
feedback = ""

for i in range(1, MAX_ROUNDS + 1):
    draft = generate(brief, feedback)
    verdict = evaluate(draft)
    print(f"--- round {i} ---")
    print("draft:   ", draft)
    print("verdict: ", verdict)
    if verdict["approved"]:
        print("\n✅ Approved.")
        break
    feedback = verdict["feedback"]
else:
    print("\n⚠️ Hit max rounds without approval. Ship the last draft or escalate.")

--- round 1 ---
draft:    Stay hydrated effortlessly with this sleek $39 smart water bottle that tracks your daily water intake and gently glows as a reminder to drink up. Perfect for busy lifestyles, it syncs with your phone to ensure you hit your hydration goals without a second thought.
verdict:  {'approved': True, 'feedback': "The copy meets the criteria by avoiding clichés, including a concrete price ($39), and adhering to the two-sentence limit. It effectively communicates the product's functionality and benefits in a concise manner."}

✅ Approved.


## Advantages

1. The generator focuses on writing.
2. The evaluator focuses on the rubric.
3. Neither has to do both jobs at once.

>  LLMs are usually better at judging a piece of text than at producing one that satisfies every constraint in one shot. The evaluator role plays to that strength.

> Use a cheaper model for the generator and a better one for the evaluator.

Two things make this **not** an agent:

- The loop shape is fixed: always generate → evaluate. The model can't decide to skip evaluation, call a new tool, or restart.
- The stopping criterion is yours: either the evaluator says approved, or you hit MAX_ROUNDS. The model never chooses to stop.
